In [1]:
!pip install -q requests beautifulsoup4 markdownify

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import time
import re
import json
from markdownify import markdownify as mdify

In [ ]:
# Удаляем старую директорию во избежании ошибок
chunks_dir = Path("./articles")
if chunks_dir.exists():
    shutil.rmtree(chunks_dir)
    print("Старая папка 'articles' удалена.")

Меню может иметь произвольную глубину вложенности. Рекурсивный обход гарантирует, что мы не пропустим ни один раздел, даже если структура изменится.

In [3]:
BASE_URL = "https://academy.lamoda.ru"
START_URL = BASE_URL + "/articles/start-working/"

def get_all_section_urls() -> list:
    resp = requests.get(START_URL, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    sidebar = soup.find('div', id='sidebar')
    if not sidebar:
        raise Exception("Боковое меню не найдено")
    urls = set()
    def traverse(ul):
        for li in ul.find_all('li', recursive=False):
            a = li.find('a', href=True)
            if a:
                full = urljoin(BASE_URL, a['href'])
                if full.startswith(BASE_URL) and '/articles/' in full:
                    urls.add(full)
            inner = li.find('ul')
            if inner:
                traverse(inner)
    top_ul = sidebar.find('ul', class_='sidebar-menu-list')
    if top_ul:
        traverse(top_ul)
    return sorted(urls)

section_urls = get_all_section_urls()
print(f"Найдено {len(section_urls)} разделов")

Найдено 82 разделов


Страницы разделов используют два типа блоков со статьями, собираем оба.
Некоторые страницы могут иметь кнопку "Показать еще", учитываем это.
Lamoda может временно блокировать частые запросы. При получении 503 делаем повторный запрос после паузы (экспоненциальная задержка). Это повышает надёжность сбора.

In [4]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_article_urls_from_section(section_url: str) -> list:
    urls = []
    current_url = section_url
    page = 1

    while True:
        try:
            resp = requests.get(current_url, headers=HEADERS, timeout=15)
            if resp.status_code == 503:
                print(f"Ошибка 503 для {current_url}, повторяю запрос")
                time.sleep(10)
                resp = requests.get(current_url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')

            # Ссылки на статьи могут быть в двух разных разделах
            found = 0
            for a in soup.find_all('a', class_=['subsections-article-item__detail-link', 'current-article-item__detail-link']):
                href = a.get('href')
                if href:
                    full = urljoin(BASE_URL, href)
                    if '/articles/' in full:
                        urls.append(full)
                        found += 1
            print(f"    Страница {page}: найдено {found} статей")

            # Проверяем наличие кнопки "Показать еще"
            load_more = soup.find('div', class_='load-more')
            if load_more and load_more.get('data-url'):
                next_relative = load_more['data-url']
                current_url = urljoin(BASE_URL, next_relative)
                page += 1
                time.sleep(0.5)
            else:
                break
        except Exception as e:
            print(f"Ошибка при обработке {current_url}: {e}")
            break

    return urls 

In [5]:
all_article_urls = set()
for idx, sec in enumerate(section_urls):
    print(f"[{idx+1}/{len(section_urls)}] {sec}")
    arts = get_article_urls_from_section(sec)
    print(f"Статей: {len(arts)}")
    for a in arts:
        all_article_urls.add(a)
    time.sleep(0.5)

all_article_urls = sorted(all_article_urls)
print(f"\nВсего уникальных статей: {len(all_article_urls)}")

[1/82] https://academy.lamoda.ru/articles/aktsii/
    Страница 1: найдено 10 статей
Статей: 10
[2/82] https://academy.lamoda.ru/articles/api/
    Страница 1: найдено 0 статей
Статей: 0
[3/82] https://academy.lamoda.ru/articles/api/1-vvedenie/
    Страница 1: найдено 4 статей
Статей: 4
[4/82] https://academy.lamoda.ru/articles/api/10-fbo/
    Страница 1: найдено 4 статей
Статей: 4
[5/82] https://academy.lamoda.ru/articles/api/11-notifikatsii/
    Страница 1: найдено 4 статей
Статей: 4
[6/82] https://academy.lamoda.ru/articles/api/12-dostavka/
    Страница 1: найдено 3 статей
Статей: 3
[7/82] https://academy.lamoda.ru/articles/api/13-vozvraty/
    Страница 1: найдено 2 статей
Статей: 2
[8/82] https://academy.lamoda.ru/articles/api/14-voprosy-o-tovarakh/
    Страница 1: найдено 1 статей
Статей: 1
[9/82] https://academy.lamoda.ru/articles/api/15-markirovka/
    Страница 1: найдено 3 статей
Статей: 3
[10/82] https://academy.lamoda.ru/articles/api/16-reshenie-problem/
    Страница 1: найдено

Собираем ТОЛЬОК статьи, убираем все лишнее. Так же на будущее, для ссылок на картинки, восстонавливем абсолютный путь.

In [6]:
def fetch_article(url: str, retries=3) -> dict:
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code == 503:
                print(f"Ошибка 503 {url}, попытка {attempt+1}")
                time.sleep(5 * (attempt + 1))
                continue
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'html.parser')

            # Заголовок H1
            h1 = soup.find('h1')
            title = h1.get_text(strip=True) if h1 else "Без заголовка"

            # Дата обновления
            date_str = ""
            date_div = soup.find('div', class_='article-detail-date__value')
            if date_div:
                date_span = date_div.find('span', class_='body-s')
                if date_span:
                    date_str = date_span.get_text(strip=True)

            # Теги
            tags = []
            for tag_span in soup.find_all('span', class_='article-detail__tag'):
                a = tag_span.find('a')
                if a:
                    tags.append(a.get_text(strip=True))

            # Основной контент
            content_div = soup.find('div', class_='article-detail__text')
            if not content_div:
                # Запасной вариант
                content_div = soup.find('div', class_='article-detail-content')
            if not content_div:
                content_div = soup.find('body')

            # Очистка мусора
            for bad in content_div.find_all(['script', 'style', 'footer', 'nav']):
                bad.decompose()
            for cls in ['article-detail-share-flex', 'article-feedback', 'article-detail-bottom',
                        'article-detail-prev-next-container', 'article-detail-page-nav',
                        'article-detail-tags-date', 'component-main-header', 'breadcrumb-title-container',
                        'back-link-container']:
                for elem in content_div.find_all(class_=cls):
                    elem.decompose()
            # Удаляем верхний заголовок H1, если он дублируется внутри контента
            for h in content_div.find_all('h1'):
                h.decompose()

            # Создаем полную ссылку на картинки
            for img in content_div.find_all('img'):
                src = img.get('src')
                if src:
                    img['src'] = urljoin(url, src)

            html_content = str(content_div)
            md_text = mdify(html_content, heading_style="ATX", strip=['script', 'style'])
            # Убираем пустые строки
            md_text = re.sub(r'\n{3,}', '\n\n', md_text).strip()

            return {
                "url": url,
                "title": title,
                "last_update": date_str,
                "tags": tags,
                "content_md": md_text
            }
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
                continue
            print(f"Ошибка {url}: {e}")
            return {}
    return {}

In [7]:
OUTPUT_DIR = "articles"
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata_list = []

for idx, url in enumerate(all_article_urls):
    print(f"[{idx+1}/{len(all_article_urls)}] {url}")
    article = fetch_article(url)
    if not article:
        continue

    # Создаем имя файла используя slug
    slug = url.rstrip('/').split('/')[-1]
    if not slug or slug == 'articles':
        parts = url.rstrip('/').split('/')
        slug = parts[-2] if len(parts) >= 2 else 'index'
    filename = f"{slug}.md"
    filepath = os.path.join(OUTPUT_DIR, filename)

    # Формируем только нужный контент в md файлах
    md_content = f"# {article['title']}\n\n{article['content_md']}"

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(md_content)

    metadata_list.append({
        "url": article['url'],
        "title": article['title'],
        "last_update": article['last_update'],
        "tags": article['tags'],
        "file": filepath
    })
    time.sleep(0.3)

print(f"\nСохранено {len(metadata_list)} статей в '{OUTPUT_DIR}'")

with open('articles_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)
print("articles_metadata.json создан")

[1/293] https://academy.lamoda.ru/articles/aktsii/aktsii-s-dopolnitelnoy-skidkoy-ot-lamoda/
[2/293] https://academy.lamoda.ru/articles/aktsii/aktsii-s-lyuboy-skidkoy/
[3/293] https://academy.lamoda.ru/articles/aktsii/instruktsiya-po-rabote-s-edinym-excel-shablonom-v-aktsiyakh/
[4/293] https://academy.lamoda.ru/articles/aktsii/kak-dobavit-tovary-v-aktsiyu-cherez-interfeys/
[5/293] https://academy.lamoda.ru/articles/aktsii/kak-udalit-tovary-iz-aktsii-cherez-interfeys/
[6/293] https://academy.lamoda.ru/articles/aktsii/pravila-izmeneniya-aktsionnoy-skidki-u-tovara-kotoryy-uzhe-uchastvuet-v-aktsii/
[7/293] https://academy.lamoda.ru/articles/aktsii/razdel-aktsii-osnovnaya-informatsiya/
[8/293] https://academy.lamoda.ru/articles/aktsii/targetirovannye-aktsii/
[9/293] https://academy.lamoda.ru/articles/aktsii/tovary-s-potentsialom-uluchsheniya-metrik-za-schet-uchastiya-v-aktsiyakh/
[10/293] https://academy.lamoda.ru/articles/aktsii/validatsiya-aktsionnykh-tsen/
[11/293] https://academy.lamoda.